### Imports

In [ ]:
import functools as ft
import pathlib
import time
import typing

import jax
import jax.experimental.sparse as jax_sprs
import jax.numpy as jnp
import matplotlib.collections as mpl_collections
import matplotlib.patches as mpl_patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.interpolate as spy_interp

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz


In [ ]:
jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

results_dir = pathlib.Path("results/paper/swell_nu")
results_dir.mkdir(parents=True, exist_ok=True)

### Targets

In [ ]:
length_scale = 1.0e3  # m to mm . The curve was gen in m. To keep in m, use 1
target_df = pd.read_csv("crops/data/shape_morph_targets/nu.csv")
target_arc = length_scale * target_df["s"].values
target_points = length_scale * target_df[["x", "y", "z"]].values
target_curve_fn = spy_interp.CubicSpline(target_arc, target_points)

rod_length = float(target_arc[-1])  # The rod length is taken from the target

### Material

In [ ]:
shear_modulus_gel, shear_modulus_elastomer = 1.0e-3, 5.0e-2  # kN/mm^2 = 10^9 N / m^2
chi_gel, chi_elastomer = 0.2, 2.5

### Solvent

In [ ]:
solvent = _fe.FlorySolvent(
  bath_temperature=310.0,
  molar_volume=1.8e4,  # #mm^3/mol
  ref_chem_potential=0.0,
)
num_steps = 70
dry_chem_potential = -1.0e5
wet_chem_potential = -100.0
t = jnp.linspace(0.0, 1.0, num_steps + 1)[1:]
ramp = t**0.03
chem_potentials = dry_chem_potential + (wet_chem_potential - dry_chem_potential) * ramp


In [ ]:
plt.plot(t, chem_potentials, "r*")

### Solver settings

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-12,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-8,
}
bisection_settings = {
  "lower": 1e-6,
  "upper": 1.0 - 1e-6,
  "tol": 1e-6,
  "max_iter": 200,
}


### Continuation schemes

In [ ]:
continuation_epochs = 60
thresh_beta_max = 128.0
grayness_start, grayness_min = 0.3, 0.02
thresh_beta_rate = (thresh_beta_max - 1.0) / continuation_epochs
grayness_rate = (grayness_start - grayness_min) / continuation_epochs

### Baseline

In [ ]:
baseline_num_elems = 300
baseline_settings = {
  "num_elems": baseline_num_elems,
  "num_fibers": 12,
  "aspect_ratio": 200.0,
  "filter_radius": 2.0 * rod_length / baseline_num_elems,
}

### problem setup

In [ ]:
class NuSetup(typing.NamedTuple):
  """Stores discretization dependent objects needed by optimization.

  Attributes:
    rod: `Rod` of the straight dry rod of length `L` along `+x`.
    problem: `CosseratProblem` holding the rod, the boundary conditions, the Newton
      solver, and the solvent.
    filter_matrix: Sparse matrix of shape `(num_elems, num_elems)` of the normalized
      linear density filter along the rod axis.
    target_nodes: Array of shape `(num_nodes, 3)` [m] of target positions sampled at
      the reference arc coordinate of every rod node.
    target_curvature: Array of shape `(num_elems, 3)` of the target Darboux vector in
      its Bishop frame multiplied by `L`, so every entry is dimensionless.
    fiber_heights: Array of shape `(num_elems, num_fibers)` [m] of layer extents
      along `d1`, used for rendering.
    fiber_thicknesses: Array of shape `(num_elems, num_fibers)` [m] of layer extents
      along `d2`, used for rendering.
    rod_height: Section height `h` [m] along `d1`.
    num_elems: Number of rod elements.
    num_fibers: Number of layers across the section height.
  """

  rod: _mesher.Rod
  problem: _fe.CosseratProblem
  filter_matrix: jax_sprs.BCOO
  target_nodes: jnp.ndarray
  target_curvature: jnp.ndarray
  fiber_heights: jnp.ndarray
  fiber_thicknesses: jnp.ndarray
  rod_height: float
  num_elems: int
  num_fibers: int

### Problem builder

In [ ]:
def build_setup(
  num_elems: int,
  num_fibers: int,
  aspect_ratio: float,
  filter_radius: float,
  use_self_contact: bool = True,
) -> NuSetup:
  """Builds the rod, boundary conditions, solver, filter, and target for one run.

  The rod takes the length `L` of the target. Its rect. cs has ht `h = L / aspect_ratio`
  along `d1` and breadth `b = 0.7 h` along `d2`, and the height is split into
  `num_fibers` layers stacked along `d1` only. Every layer therefore sits at
  `y2 = 0`. This supports bending in the `x-y` plane through `κ2`, but the offsets
  provide no out-of-plane bending stiffness, so the translation `Z` and rotation
  `θ_Y` are fixed at every node to keep the tangent matrix nonsingular.

  Algorithm.
    1. Set `h = L / aspect_ratio`, `b = 0.7 h`, and the layer height
       `h_f = h / num_fibers`.
    2. Place the layer centroids at `y1_k = −(h − h_f)/2 + k h_f` for
       `k = 0, …, num_fibers − 1`, with area `A_k = h_f b` and centroidal second
       moments `h_f b³ / 12` about `d1` and `b h_f³ / 12` about `d2`.
    3. Build a straight rod along `+x` with `num_elems` elements of reference length
       `L / num_elems`, with the seed binormal along `+z` so that `d2 = e_z`.
    4. Clamp all six dofs of node 0, and fix `Z` and `θ_Y` at every node, for all
       `num_steps` chemical potential increments.
    5. Assemble the Newton solver and the swelling problem.
    6. Build the linear density filter along the axis with cutoff `R` in physical
       length, so refining the mesh leaves the regularization unchanged.
    7. Sample the target at the rod nodes, compute its Bishop-frame Darboux vector
       per unit target arc length, and multiply it by `L`.

  Args:
    num_elems: Number of rod elements, positive. Every fillet of the target should
      span at least about 8 elements.
    num_fibers: Number of layers across the section height, positive, typically
      8 to 24.
    aspect_ratio: Slenderness `L / h`, positive, typically 100 to 600 for the NU
      target.
    filter_radius: Cutoff `R` [m] of the linear density filter along the axis,
      positive. A value below one element length leaves the design unfiltered.

  Returns:
    A `NuSetup` holding every object one optimization run needs.
  """
  # 1. Mesh overall
  num_nodes = num_elems + 1
  rod_height = rod_length / aspect_ratio
  rod_breadth = 0.7 * rod_height
  layer_height = rod_height / num_fibers

  # 2. Fibers
  offsets_d1 = np.linspace(
    -0.5 * (rod_height - layer_height), 0.5 * (rod_height - layer_height), num_fibers
  )
  fiber_offsets = jnp.zeros((num_elems, num_fibers, 2)).at[:, :, 0].set(offsets_d1)
  fiber_centroid_moi = jnp.stack(
    [
      jnp.full((num_elems, num_fibers), layer_height * rod_breadth**3 / 12.0),
      jnp.full((num_elems, num_fibers), rod_breadth * layer_height**3 / 12.0),
    ],
    axis=-1,
  )
  parametric_coords = jnp.linspace(0.0, rod_length, num_nodes)

  # 3. rod
  rod = _mesher.build_rod_geometry(
    analytical_curve=lambda arc: jnp.array([arc, 0.0, 0.0]),
    parametric_coords=parametric_coords,
    fiber_offsets=fiber_offsets,
    fiber_areas=jnp.full((num_elems, num_fibers), layer_height * rod_breadth),
    fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
    fiber_centroid_moi=fiber_centroid_moi,
    seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  )

  # 4. BC
  clamped_dofs = (
    _utils.Translation.X,
    _utils.Translation.Y,
    _utils.Translation.Z,
    _utils.Rotation.THETA_X,
    _utils.Rotation.THETA_Y,
    _utils.Rotation.THETA_Z,
  )
  clamp = [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[np.array([0]) for _ in range(num_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ]
  planar = [
    _bc.DirichletBC(
      values=(dof, [np.zeros(num_nodes) for _ in range(num_steps)]),
      node_targets=[np.arange(num_nodes) for _ in range(num_steps)],
      name=f"planar_{dof}",
    )
    for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
  ]
  bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=clamp + planar)

  # 4b. Self contact
  self_contact = None
  if use_self_contact:
    contact_radius = float(jnp.max(rod.contact_radii))
    elem_length = rod_length / num_elems
    self_contact = _contact.SelfContact.from_rod(
      rod=rod,
      stiffness=_contact.compute_self_contact_stiffness(
        rod=rod,
        youngs_modulus=3.0 * shear_modulus_elastomer,
        stiffness_ratio=0.5,
      ),
      adjacency_exclusion=int(np.ceil(2.0 * contact_radius / elem_length)) + 2,
      smoothing_fraction=0.05,
      max_candidate_pairs=4000,
    )
  # 5. Problem
  problem = _fe.CosseratProblem(
    rod=rod,
    bc=bc,
    nr_solver=_solver.NewtonRaphson(
      solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
      nonlinear_settings=nonlinear_settings,
      linear_settings=linear_settings,
      verbose=True,
    ),
    solvent=solvent,
    bisection_settings=bisection_settings,
    self_contact=self_contact,
  )

  # 6. filter
  arc_centers = 0.5 * (rod.arc_coords[:-1] + rod.arc_coords[1:])
  filter_matrix = _utils.create_density_filter(
    coords=arc_centers[:, None], cutoff_distance=filter_radius
  )

  # 7. Target
  target_nodes = jnp.asarray(target_curve_fn(np.asarray(parametric_coords)))
  target_kinematics = _mesher.compute_curvature_from_centerline(
    node_coords=target_nodes, seed_binormal=jnp.array([0.0, 0.0, 1.0])
  )

  # 8. Return
  return NuSetup(
    rod=rod,
    problem=problem,
    filter_matrix=filter_matrix,
    target_nodes=target_nodes,
    target_curvature=target_kinematics.curvature * rod_length,
    fiber_heights=jnp.full((num_elems, num_fibers), layer_height),
    fiber_thicknesses=jnp.full((num_elems, num_fibers), rod_breadth),
    rod_height=rod_height,
    num_elems=num_elems,
    num_fibers=num_fibers,
  )


### Objective

In [ ]:
def compute_pseudodensity(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  setup: NuSetup,
) -> jnp.ndarray:
  """Maps the flat design vector to the filtered and projected layer pseudodensity.

  Each layer is filtered independently along the rod axis by the normalized linear
  filter `W`, giving `ρ̃_{e,k} = Σ_j W_{ej} x_{j,k}`. The result is then projected
  toward zero or one by the smoothed Heaviside function
  `ρ̂ = [tanh(βη) + tanh(β(ρ̃ − η))] / [tanh(βη) + tanh(β(1 − η))]` with `η = 1/2`.
  A value of one is gel and a value of zero is elastomer.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`, ordered element by element.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    setup: `NuSetup` supplying the discretization and the filter.

  Returns:
    Array of shape `(num_elems, num_fibers)` of pseudodensity values in `[0, 1]`.
  """
  design = design_vars.reshape(setup.num_elems, setup.num_fibers)
  filtered = setup.filter_matrix @ design
  return _utils.threshold_filter(density=filtered, beta=thresh_beta)


def compute_objective(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  setup: NuSetup,
) -> tuple[jnp.ndarray, jnp.ndarray, tuple[jnp.ndarray, jnp.ndarray]]:
  """Computes the log relative curvature mismatch, its gradient, and the solved state.

  The ratio inside the logarithm is one for any layout that stays straight, which
  includes the uniform initial design. The objective therefore starts at zero and
  decreases, and its exponential reads as the fraction of the straight-rod mismatch
  that remains. The logarithm keeps the gradient `∇J = ∇r / r` of order one while the
  ratio `r` falls by orders of magnitude, so the objective is never rescaled. All
  three Darboux components enter, so spurious out-of-plane bending or twist is
  penalized. The value and gradient are returned in the shapes MMA expects, together
  with the solved state, so post-processing needs no extra solve.

  Algorithm.
    1. Map the design vector to the pseudodensity `ρ̂` by filtering and projection.
    2. Interpolate `G = ρ̂ G_gel + (1 − ρ̂) G_el` and `χ = ρ̂ χ_gel + (1 − ρ̂) χ_el` in
       every layer of every element.
    3. Solve the swelling equilibrium along the chemical potential ramp from
       `dry_chem_potential` to `wet_chem_potential`.
    4. Recover the deformed nodes and directors, and compute the Darboux vector `κ_e`
       of every element per unit reference arc length.
    5. Evaluate `J = log( Σ_e ‖κ_e L − κ̂_e L‖² / Σ_e ‖κ̂_e L‖² )`, where `κ̂_e` is the
       target curvature.
    6. Differentiate steps 1 to 5 in reverse mode with respect to the design vector.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    setup: `NuSetup` supplying the rod, the problem, the filter, and the target.

  Returns:
    A tuple `(objective, grad_objective, (pseudodensity, u_star))`. `objective` has
      shape `(1,)` and equals zero for a straight rod. `grad_objective` has shape
      `(num_design_var, 1)`. `pseudodensity` has shape `(num_elems, num_fibers)`.
      `u_star` has shape `(num_dofs,)` [m, rad] and holds the converged displacements
      and rotations.
  """

  def objective_fn(
    design_vars: jnp.ndarray,
  ) -> tuple[jnp.ndarray, tuple[jnp.ndarray, jnp.ndarray]]:
    """Evaluates `J` and returns the pseudodensity and solved state as auxiliary."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars,
      thresh_beta=thresh_beta,
      setup=setup,
    )
    shear_moduli = shear_modulus_gel * pseudodensity + shear_modulus_elastomer * (
      1.0 - pseudodensity
    )
    flory_parameters = chi_gel * pseudodensity + chi_elastomer * (1.0 - pseudodensity)

    u_star, _ = setup.problem.solve(
      u0=jnp.zeros(setup.rod.num_dofs),
      shear_moduli=shear_moduli,
      flory_parameters=flory_parameters,
      chem_potentials=chem_potentials,
    )
    coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
      rod=setup.rod,
      u=u_star,
    )
    kinematics = _mesher.compute_discrete_cosserat_kinematics(
      node_coords=coords,
      directors=directors,
      arc_coords=setup.rod.arc_coords,
    )
    error = kinematics.curvature * rod_length - setup.target_curvature
    mismatch = jnp.einsum("ec,ec->", error, error)
    reference = jnp.einsum("ec,ec->", setup.target_curvature, setup.target_curvature)
    return mismatch / reference, (pseudodensity, u_star)

  (objective, aux), grad_objective = jax.value_and_grad(objective_fn, has_aux=True)(
    design_vars
  )
  return objective.reshape(1), grad_objective.reshape(-1, 1), aux


### Constraints

In [ ]:
def compute_constraints(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  max_grayness: float,
  max_gel_fraction: float,
  setup: NuSetup,
) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Computes the normalized grayness and gel fraction constraints and their gradients.

  Algorithm.
    1. Map the design vector to the pseudodensity `ρ̂` by filtering and projection.
    2. Evaluate the grayness constraint `mean(ρ̂ (1 − ρ̂)) / γ_max − 1`. The grayness is
       zero for a two-phase design and one quarter for a uniform design of one half.
    3. Evaluate the gel fraction constraint `mean(ρ̂) / V_max − 1`.
    4. Differentiate each constraint separately in reverse mode with respect to the
       design vector, and stack the values and gradients in the shapes MMA expects.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    max_grayness: Permitted grayness `γ_max`, in `(0, 0.25]`, annealed toward a
      small final value.
    max_gel_fraction: Permitted gel volume fraction `V_max`, in `(0, 1]`.
    setup: `NuSetup` supplying the discretization and the filter.

  Returns:
    A tuple `(constraints, grad_constraints)`. `constraints` has shape
      `(num_cons=2, 1)` and holds the grayness constraint followed by the gel
      fraction constraint. `grad_constraints` has shape `(num_cons=2, num_design_var)`.
  """

  def grayness_fn(design_vars: jnp.ndarray) -> jnp.ndarray:
    """Evaluates the normalized grayness constraint `γ / γ_max − 1`."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars,
      thresh_beta=thresh_beta,
      setup=setup,
    )
    return jnp.mean(pseudodensity * (1.0 - pseudodensity)) / max_grayness - 1.0

  def gel_fraction_fn(design_vars: jnp.ndarray) -> jnp.ndarray:
    """Evaluates the normalized gel fraction constraint `V / V_max − 1`."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars,
      thresh_beta=thresh_beta,
      setup=setup,
    )
    return jnp.mean(pseudodensity) / max_gel_fraction - 1.0

  grayness, grad_grayness = jax.value_and_grad(grayness_fn)(design_vars)
  gel_fraction, grad_gel_fraction = jax.value_and_grad(gel_fraction_fn)(design_vars)
  return (
    jnp.stack([grayness, gel_fraction]).reshape(-1, 1),
    jnp.stack([grad_grayness, grad_gel_fraction]),
  )

### Plotting

In [ ]:
in_plane = _mesher.CurvatureComponent.BENDING_D2
axial = _mesher.StrainComponent.AXIAL_STRETCH
target_color = "tab:green"


def compute_kinematics(setup: NuSetup, disp: np.ndarray) -> _mesher.CosseratKinematics:
  """Computes the element strains and curvatures of the rod deformed by `disp`.

  Args:
    setup: `NuSetup` of the discretization the state belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    The `CosseratKinematics` of every element, per unit reference arc length.
  """
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=setup.rod, u=jnp.asarray(disp)
  )
  return _mesher.compute_discrete_cosserat_kinematics(
    node_coords=coords, directors=directors, arc_coords=setup.rod.arc_coords
  )


def compute_rod_cells(
  setup: NuSetup,
  disp: np.ndarray,
  thickness_scale: float = 4.0,
) -> tuple[np.ndarray, np.ndarray]:
  """Computes the in-plane quadrilateral of every element-layer cell of the rod.

  Algorithm.
    1. Recover the deformed nodes `r_n` and directors from `disp`, and keep the
       in-plane components of `d₁,n`.
    2. Place the layer edges at `y_j = s_t h (j / num_fibers − 1/2)` for
       `j = 0, …, num_fibers`, where `s_t` is the thickness magnification, and form
       the corner points `p_{n,j} = r_n + y_j d₁,n`.
    3. Form cell `(e, k)` as the quadrilateral
       `(p_{e,k}, p_{e+1,k}, p_{e+1,k+1}, p_{e,k+1})`.
    4. Trace the outline forward along the `j = 0` edge and back along the
       `j = num_fibers` edge.

  Args:
    setup: `NuSetup` of the discretization the state belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    thickness_scale: Magnification `s_t` of the drawn section thickness, positive.

  Returns:
    A tuple `(cells, outline)`. `cells` has shape `(num_elems * num_fibers, 4, 2)`
      [mm] and is ordered element by element like the design. `outline` has shape
      `(2 * num_nodes, 2)` [mm] and is the closed boundary of the rod.
  """
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=setup.rod, u=jnp.asarray(disp)
  )
  centerline = np.asarray(coords)[:, :2]
  in_plane_d1 = np.asarray(directors)[:, 0, :2]
  layer_edges = (
    thickness_scale * setup.rod_height * np.linspace(-0.5, 0.5, setup.num_fibers + 1)
  )
  corners = centerline[:, None, :] + np.einsum("j,nd->njd", layer_edges, in_plane_d1)
  cells = np.stack(
    [corners[:-1, :-1], corners[1:, :-1], corners[1:, 1:], corners[:-1, 1:]], axis=2
  ).reshape(-1, 4, 2)
  outline = np.concatenate([corners[:, 0], corners[::-1, -1]])
  return cells, outline


def plot_convergence(
  history: dict[str, np.ndarray],
  max_gel_fraction: float,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the loss and the two normalized constraints against the iteration.

  The left axis shows the loss on a logarithmic scale. The right axis shows the gel
  constraint `V / V_max − 1` and the grayness constraint `γ / γ_max,k − 1`, both
  feasible at or below the dotted zero line. The history stores the raw gel fraction
  `V = mean(ρ̂)` and grayness `γ = mean(ρ̂ (1 − ρ̂))`, so the grayness bound
  `γ_max,k` is rebuilt from the continuation schedule of `run_optimization`.

  Args:
    history: Dictionary returned by `run_optimization` or loaded from its `.npz`,
      providing `epoch`, `objective`, `grayness`, and `gel_fraction`, each of shape
      `(num_epochs,)`.
    max_gel_fraction: Permitted gel volume fraction `V_max` in `(0, 1]` that the
      history was optimized with.

  Returns:
    A tuple `(fig, ax)` holding the figure and the loss axis. The constraint axis is
      its twin.
  """
  epoch = np.asarray(history["epoch"])
  max_grayness = np.maximum(grayness_min, grayness_start - grayness_rate * epoch)

  fig, ax = plt.subplots(figsize=(7.0, 4.0))
  constraint_ax = ax.twinx()
  lines = ax.semilogy(epoch, history["objective"], color="k", ls="-", label="Loss")
  lines += constraint_ax.plot(
    epoch,
    history["gel_fraction"] / max_gel_fraction - 1.0,
    color="C0",
    ls="--",
    label="Gel constraint",
  )
  lines += constraint_ax.plot(
    epoch,
    history["grayness"] / max_grayness - 1.0,
    color="C1",
    ls="-.",
    label="Grayness constraint",
  )
  constraint_ax.axhline(0.0, color="0.6", ls=":", lw=0.8)
  ax.set_xlim(epoch[0], epoch[-1])
  ax.set_xlabel("Iteration")
  ax.set_ylabel("Loss")
  constraint_ax.set_ylabel("Constraint")
  ax.legend(
    handles=lines,
    frameon=False,
    ncol=3,
    handlelength=1.6,
    columnspacing=1.0,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
  )
  fig.tight_layout()
  return fig, ax


def plot_rod_3d(
  setup: NuSetup,
  design: np.ndarray,
  disp: typing.Optional[np.ndarray] = None,
  thickness_scale: float = 4.0,
  view: tuple[float, float] = (25.0, -60.0),
  ax: typing.Optional[plt.Axes] = None,
) -> plt.Axes:
  """Draws the rod as solid 3D layer cells colored by gel or elastomer.

  Every element-layer cell is a box spanned by the node directors, as in
  `_viz.plot_rod_fiber_sections`. The undeformed rod is always drawn translucent.
  When `disp` is given the swollen rod is drawn opaque over it. The section height
  and breadth are magnified by `thickness_scale` in this drawing only.

  Args:
    setup: `NuSetup` of the discretization the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`,
      one for gel and zero for elastomer.
    disp: Optional array of shape `(num_dofs,)` [m, rad] of converged displacements
      and rotations. If None, only the undeformed rod is drawn.
    thickness_scale: Magnification `s_t` of the drawn section, positive.
    view: `(elev, azim)` of the camera in degrees.
    ax: Optional Matplotlib 3D axis. If None, one is created.

  Returns:
    The Matplotlib 3D axis.
  """
  if ax is None:
    ax = plt.figure(figsize=(7.0, 5.0)).add_subplot(projection="3d")
  layer_height = thickness_scale * setup.rod_height / setup.num_fibers
  offsets_d1 = (thickness_scale * setup.rod_height - layer_height) * np.linspace(
    -0.5, 0.5, setup.num_fibers
  )

  states = [(jnp.zeros(setup.rod.num_dofs), 0.3)]
  if disp is not None:
    states.append((jnp.asarray(disp), 1.0))
  for u, alpha in states:
    coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
      rod=setup.rod, u=u
    )
    coords, directors = np.asarray(coords), np.asarray(directors)
    fiber_coords = coords[:, None, :] + np.einsum(
      "f,nd->nfd", offsets_d1, directors[:, 0, :]
    )
    _viz.plot_rod_fiber_sections(
      fiber_coordinates=fiber_coords,
      directors=directors,
      fiber_heights=np.full(np.shape(design), layer_height),
      fiber_thicknesses=thickness_scale * np.asarray(setup.fiber_thicknesses),
      fiber_field=np.asarray(design),
      fiber_field_val_range=(0.0, 1.0),
      cmap=_viz.elast_gel_cmap,
      alpha=alpha,
      edge_width=0.0,
      ax=ax,
    )

  ax.view_init(elev=view[0], azim=view[1])
  ax.set_box_aspect((1.0, 1.0, 1.0))
  ax.set_xlabel(r"$x$")
  ax.set_ylabel(r"$y$")
  ax.set_zlabel(r"$z$")
  ax.legend(
    handles=[
      mpl_patches.Patch(color=_viz.gel_colors["hydrogel"], label="Gel"),
      mpl_patches.Patch(color=_viz.gel_colors["elastomer"], label="Elastomer"),
    ],
    frameon=False,
    loc="upper right",
  )
  return ax


def plot_swollen_shape(
  setup: NuSetup,
  design: np.ndarray,
  disp: np.ndarray,
  thickness_scale: float = 4.0,
) -> tuple[plt.Figure, plt.Axes]:
  """Draws the swollen rod colored by gel or elastomer over the scaled target.

  Every element-layer cell is drawn as a quadrilateral in the bending plane, colored
  by its pseudodensity, over the target scaled about the clamp by the mean stretch
  `ν̄`. The objective matches shape rather than size, so the scaled target is the
  correct reference. The section thickness is magnified by `thickness_scale` in this
  drawing only, so that the layers remain visible at a slenderness of several
  hundred. The bending plane is the computational `x-y` plane, labeled `x-z` here.

  Args:
    setup: `NuSetup` of the discretization the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    thickness_scale: Magnification `s_t` of the drawn section thickness, positive,
      typically 1 to 8. It affects the drawing only.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  cells, _ = compute_rod_cells(setup=setup, disp=disp, thickness_scale=thickness_scale)
  stretch = np.asarray(compute_kinematics(setup=setup, disp=disp).strain[:, axial])
  scaled_target = float(np.mean(stretch)) * np.asarray(setup.target_nodes)

  fig, ax = plt.subplots(figsize=(6.5, 3.0))
  rod_cells = mpl_collections.PolyCollection(
    verts=cells, cmap=_viz.elast_gel_cmap, edgecolors="none"
  )
  rod_cells.set_array(np.asarray(design).ravel())
  rod_cells.set_clim(0.0, 1.0)
  ax.add_collection(rod_cells)
  ax.plot(scaled_target[:, 0], scaled_target[:, 1], color="k", ls="--", lw=0.8)
  ax.set_aspect("equal")
  ax.set_xlabel(r"$x$ (mm)")
  ax.set_ylabel(r"$z$ (mm)")
  ax.spines[["top", "right"]].set_visible(False)
  fig.tight_layout()
  return fig, ax


def plot_curvature(setup: NuSetup, disp: np.ndarray) -> tuple[plt.Figure, plt.Axes]:
  """Plots the optimized against the target in-plane curvature `κ₂ L`.

  The two curves nearly coincide, so the target is drawn green and dashed over a
  soft glow of three widening translucent strokes, and the optimized curve is drawn
  solid black between the glow and the dashes.

  Args:
    setup: `NuSetup` of the discretization the state belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  curvature = rod_length * np.asarray(
    compute_kinematics(setup=setup, disp=disp).curvature[:, in_plane]
  )
  target = np.asarray(setup.target_curvature[:, in_plane])
  arc_coords = np.asarray(setup.rod.arc_coords)
  elem_arc = 0.5 * (arc_coords[:-1] + arc_coords[1:]) / rod_length

  fig, ax = plt.subplots(figsize=(6.5, 3.0))
  glow = tuple(
    ax.step(
      elem_arc,
      target,
      where="mid",
      color=target_color,
      lw=lw,
      alpha=0.15,
      solid_joinstyle="round",
      solid_capstyle="round",
    )[0]
    for lw in (9.0, 6.0, 3.5)
  )
  (optimized_line,) = ax.step(elem_arc, curvature, where="mid", color="k", lw=1.0)
  (target_line,) = ax.step(
    elem_arc, target, where="mid", color=target_color, ls="--", lw=1.5
  )
  ax.set_xlim(0.0, 1.0)
  ax.set_xlabel(r"Normalized arc length $s/L$")
  ax.set_ylabel(r"$\kappa_2 L$")
  ax.legend(
    handles=[(*glow, target_line), optimized_line],
    labels=["Target", "Optimized"],
    frameon=False,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
  )
  fig.tight_layout()
  return fig, ax


def plot_layer_layout(
  setup: NuSetup, design: np.ndarray
) -> tuple[plt.Figure, plt.Axes]:
  """Draws the pseudodensity of every layer over the reference arc length.

  Wherever `κ₂ L > 0` the gel should sit at `y₁ < 0`, because the swelling side
  becomes the convex side of the bend.

  Args:
    setup: `NuSetup` of the discretization the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  half_height = 0.5 * setup.rod_height
  fig, ax = plt.subplots(figsize=(6.5, 2.4))
  ax.imshow(
    np.asarray(design).T,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=(0.0, rod_length, -half_height, half_height),
    cmap=_viz.elast_gel_cmap,
    vmin=0.0,
    vmax=1.0,
  )
  ax.set_xlabel(r"Arc length $s$ (mm)")
  ax.set_ylabel(r"$z$ (mm)")
  fig.tight_layout()
  return fig, ax


def plot_axial_stretch(setup: NuSetup, disp: np.ndarray) -> tuple[plt.Figure, plt.Axes]:
  """Plots the axial stretch `ν(s)` and its mean `ν̄`.

  Args:
    setup: `NuSetup` of the discretization the state belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  stretch = np.asarray(compute_kinematics(setup=setup, disp=disp).strain[:, axial])
  arc_coords = np.asarray(setup.rod.arc_coords)
  elem_arc = 0.5 * (arc_coords[:-1] + arc_coords[1:]) / rod_length

  fig, ax = plt.subplots(figsize=(6.5, 2.4))
  ax.plot(elem_arc, stretch, color="k", lw=1.0)
  ax.axhline(float(np.mean(stretch)), color="0.55", ls="--", lw=0.8)
  ax.set_xlim(0.0, 1.0)
  ax.set_xlabel(r"Normalized arc length $s/L$")
  ax.set_ylabel(r"$\nu$")
  fig.tight_layout()
  return fig, ax


def plot_final_state(
  setup: NuSetup,
  design: np.ndarray,
  disp: np.ndarray,
  thickness_scale: float = 4.0,
) -> dict[str, tuple[plt.Figure, plt.Axes]]:
  """Draws the swollen shape, curvature, layer layout, and axial stretch as figures.

  Args:
    setup: `NuSetup` of the discretization the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    thickness_scale: Magnification `s_t` of the drawn section thickness in the
      shape figure, positive.

  Returns:
    A dictionary of `(fig, ax)` pairs keyed by `shape`, `curvature`, `layout`, and
      `stretch`, in that order.
  """
  return {
    "shape": plot_swollen_shape(
      setup=setup, design=design, disp=disp, thickness_scale=thickness_scale
    ),
    "curvature": plot_curvature(setup=setup, disp=disp),
    "layout": plot_layer_layout(setup=setup, design=design),
    "stretch": plot_axial_stretch(setup=setup, disp=disp),
  }


def plot_design_evolution(
  setup: NuSetup,
  history: dict[str, np.ndarray],
  fractions: typing.Sequence[float] = (0.0, 0.2, 0.4, 0.6, 1.0),
  thickness_scale: float = 4.0,
  width: float = 7.0,
) -> dict[int, tuple[plt.Figure, plt.Axes]]:
  """Draws the swollen rod at several fractions of the run, one figure per snapshot.

  Every snapshot is colored by its pseudodensity, with a thin outline so that the
  uniform initial design, which maps to the white middle of the colormap, stays
  visible. Each axis fills its figure and has no ticks, spines, or margins, so save
  with `pad_inches=0`. All figures share one physical scale, so the snapshots can be
  placed side by side and compared directly.

  Algorithm.
    1. Map every fraction `f` to the stored snapshot whose epoch is closest to
       `f` times the last epoch.
    2. Compute the cells and outline of every snapshot.
    3. Set the scale so that the widest snapshot spans `width` inches, and size
       every figure to the bounding box of its snapshot at that scale.

  Args:
    setup: `NuSetup` of the discretization the history belongs to.
    history: Dictionary returned by `run_optimization` or loaded from its `.npz`,
      providing `epoch`, `saved_epoch`, `design`, and `disp`.
    fractions: Fractions in `[0, 1]` of the last epoch to draw.
    thickness_scale: Magnification `s_t` of the drawn section thickness, positive.
    width: Width in inches of the widest snapshot's figure, positive.

  Returns:
    A dictionary of `(fig, ax)` pairs keyed by the stored epoch drawn, in the order
      of `fractions`. Fractions that map to the same epoch share one figure.
  """
  saved_epoch = np.asarray(history["saved_epoch"])
  indices = dict.fromkeys(
    int(np.argmin(np.abs(saved_epoch - f * history["epoch"][-1]))) for f in fractions
  )
  snapshots = {
    int(saved_epoch[index]): (
      *compute_rod_cells(
        setup=setup, disp=history["disp"][index], thickness_scale=thickness_scale
      ),
      history["design"][index],
    )
    for index in indices
  }
  scale = width / max(np.ptp(outline[:, 0]) for _, outline, _ in snapshots.values())

  figures = {}
  for epoch, (cells, outline, design) in snapshots.items():
    (x_min, y_min), (x_max, y_max) = outline.min(axis=0), outline.max(axis=0)
    fig = plt.figure(figsize=(scale * (x_max - x_min), scale * (y_max - y_min)))
    ax = fig.add_axes((0.0, 0.0, 1.0, 1.0))
    rod_cells = mpl_collections.PolyCollection(
      verts=cells, cmap=_viz.elast_gel_cmap, edgecolors="none"
    )
    rod_cells.set_array(np.asarray(design).ravel())
    rod_cells.set_clim(0.0, 1.0)
    ax.add_collection(rod_cells)
    ax.add_patch(
      mpl_patches.Polygon(
        outline, closed=True, fill=False, ec="0.35", lw=0.4, clip_on=False
      )
    )
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect("equal")
    ax.axis("off")
    figures[epoch] = (fig, ax)
  return figures

### MMA optimization

In [ ]:
def run_optimization(
  setup: NuSetup,
  max_gel_fraction: float,
  max_iter: int = 251,
  move_limit: float = 5e-4,
  save_interval: int = 5,
  monitor: bool = False,
) -> dict[str, np.ndarray]:
  """Runs MMA with synchronized projection and grayness continuation.

  The objective and constraints are bound to one setup and compiled once, so their
  inner functions and gradients are traced a single time per run. The history is
  returned as a dictionary of arrays that can be saved directly with `np.savez`.
  Converting every epoch's outputs to NumPy for MMA forces the device to finish, so
  the recorded wall time per epoch is exact. The first entry of `time_epoch` includes
  compilation, so the compile time can be estimated as
  `time_epoch[0] − median(time_epoch[1:])`.

  Algorithm.
    1. Bind the setup into the objective and the constraints and compile both.
    2. Start from the uniform design `x = 1/2`, which has zero natural curvature, so
       the objective starts at `log 1 = 0`.
    3. At epoch `k` set `β_k = min(β_max, 1 + k (β_max − 1) / K)` and
       `γ_max,k = max(γ_min, γ_0 − k (γ_0 − γ_min) / K)`, where `K` is
       `continuation_epochs`, so both reach their final values together.
    4. Evaluate the objective, the constraints, and their gradients, and update the
       design with one MMA step, timing the whole epoch.
    5. Record the objective, grayness, gel fraction, and epoch time every epoch. Every
       `save_interval` epochs and at the last epoch, also record the pseudodensity
       and the displacement, and draw the state if `monitor` is set.
    6. Stop when MMA reports convergence after `continuation_epochs`, or at
       `max_iter`.

  Args:
    setup: `NuSetup` of the discretization to optimize.
    max_gel_fraction: Permitted gel volume fraction `V_max` in `(0, 1]`. A value of
      one disables the constraint.
    max_iter: Maximum number of MMA epochs, larger than `continuation_epochs`.
    move_limit: MMA move limit per epoch, in `(0, 1]`.
    save_interval: Number of epochs between stored snapshots, positive.
    monitor: Whether to draw the full state figure at every stored snapshot.

  Returns:
    A dictionary of arrays. `epoch`, `objective`, `grayness`, `gel_fraction`, and
      `time_epoch` have shape `(num_epochs,)`, with times in seconds. `saved_epoch`
      has shape `(num_saved,)`. `design` has shape
      `(num_saved, num_elems, num_fibers)`. `disp` has shape `(num_saved, num_dofs)`.
  """
  objective_and_grad = jax.jit(ft.partial(compute_objective, setup=setup))
  constraints_and_grad = jax.jit(ft.partial(compute_constraints, setup=setup))

  num_design_var = setup.num_elems * setup.num_fibers
  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-6,
    step_tol=1e-6,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=2,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(
    init_design_var=np.full((num_design_var, 1), 0.5), mma_params=mma_params
  )
  keys = (
    "epoch",
    "objective",
    "grayness",
    "gel_fraction",
    "time_epoch",
    "saved_epoch",
    "design",
    "disp",
  )
  history = {key: [] for key in keys}

  while mma_state.epoch < max_iter and not (
    mma_state.is_converged and mma_state.epoch > continuation_epochs
  ):
    epoch = mma_state.epoch
    thresh_beta = min(thresh_beta_max, 1.0 + thresh_beta_rate * epoch)
    max_grayness = max(grayness_min, grayness_start - grayness_rate * epoch)
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    epoch_start = time.perf_counter()
    objective, grad_objective, (pseudodensity, u_star) = objective_and_grad(
      design_vars, thresh_beta=thresh_beta
    )
    constraints, grad_constraints = constraints_and_grad(
      design_vars,
      thresh_beta=thresh_beta,
      max_grayness=max_grayness,
      max_gel_fraction=max_gel_fraction,
    )
    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.asarray(objective, dtype=np.float64),
      grad_obj=np.asarray(grad_objective, dtype=np.float64),
      cons=np.asarray(constraints, dtype=np.float64),
      grad_cons=np.asarray(grad_constraints, dtype=np.float64),
    )
    time_epoch = time.perf_counter() - epoch_start

    layout = np.asarray(pseudodensity)
    rel_error = np.exp(float(objective[0]))
    history["epoch"].append(epoch)
    history["objective"].append(float(objective[0]))
    history["grayness"].append(float(np.mean(layout * (1.0 - layout))))
    history["gel_fraction"].append(float(np.mean(layout)))
    history["time_epoch"].append(time_epoch)
    if epoch % save_interval == 0:
      history["saved_epoch"].append(epoch)
      history["design"].append(layout)
      history["disp"].append(np.asarray(u_star))
      print(
        f"epoch {epoch:4d}  rel. error {rel_error:.3e}  "
        f"gray {history['grayness'][-1]:.3f}  gel {history['gel_fraction'][-1]:.3f}  "
        f"{time_epoch:.2f} s"
      )
      if monitor:
        figures = plot_final_state(setup=setup, design=layout, disp=np.asarray(u_star))
        fig, ax = figures["shape"]
        ax.set_title(f"Epoch {epoch}, relative error {rel_error:.2e}")
        fig.tight_layout()
        plt.show()
        plt.pause(1e-6)

  if history["saved_epoch"][-1] != epoch:
    history["saved_epoch"].append(epoch)
    history["design"].append(layout)
    history["disp"].append(np.asarray(u_star))
  return {key: np.asarray(value) for key, value in history.items()}

### Baseline optimization

In [ ]:
def run_baseline():
  baseline = build_setup(**baseline_settings)
  gel_fraction_slack = 0.85
  unconstrained = run_optimization(setup=baseline, max_gel_fraction=1.0, monitor=True)
  np.savez(
    results_dir / "baseline_unconstrained.npz", **unconstrained, **baseline_settings
  )
  max_gel_fraction = gel_fraction_slack * float(unconstrained["gel_fraction"][-1])

  epoch_time = np.median(unconstrained["time_epoch"][1:])
  compile_time = unconstrained["time_epoch"][0] - epoch_time
  print(f"final relative error       = {np.exp(unconstrained['objective'][-1]):.3e}")
  print(f"unconstrained gel fraction = {unconstrained['gel_fraction'][-1]:.3f}")
  print(f"gel fraction bound         = {max_gel_fraction:.3f}")
  print(f"compile {compile_time:.1f} s, epoch {epoch_time:.2f} s")

  fig, _ = plot_convergence(history=unconstrained, max_gel_fraction=1.0)
  fig.savefig(results_dir / "baseline_unconstrained_convergence.pdf")
  figures = plot_final_state(
    setup=baseline, design=unconstrained["design"][-1], disp=unconstrained["disp"][-1]
  )
  for name, (fig, _) in figures.items():
    fig.savefig(results_dir / f"baseline_unconstrained_{name}.pdf")
  plt.show()


# run_baseline()


In [ ]:
aspect_ratio = 200.0
max_gel_fraction = 0.4

settings = {**baseline_settings, "aspect_ratio": aspect_ratio}
setup = build_setup(**settings)

In [ ]:


# history = run_optimization(
#   setup=setup,
#   max_gel_fraction=max_gel_fraction,
#   monitor=True,
# )
# np.savez(results_dir / f"aspect_ratio_{aspect_ratio:g}.npz", **history, **settings)


In [ ]:
history = np.load(results_dir / "ar200/aspect_ratio_200.npz")

In [ ]:
len(history["design"])

In [ ]:
final_design = history["design"][-1]
final_disp = history["disp"][-1]

In [ ]:
fig = plt.figure(figsize=(12.0, 5.0))
ax_first = fig.add_subplot(1, 2, 1, projection="3d")
ax_last = fig.add_subplot(1, 2, 2, projection="3d")

plot_rod_3d(setup, design=history["design"][0], ax=ax_first)
ax_first.set_title(f"Epoch {history['saved_epoch'][0]} (undeformed)")

plot_rod_3d(setup, design=history["design"][-1], disp=history["disp"][-1], ax=ax_last)
ax_last.set_title(f"Epoch {history['saved_epoch'][-1]}: undeformed + swollen")

fig.tight_layout()
fig.savefig(results_dir / f"aspect_ratio_{aspect_ratio:g}_3d.pdf")
plt.show()


In [ ]:
figures = plot_final_state(setup=setup, design=final_design, disp=np.asarray(final_disp))
for name, (fig, _) in figures.items():
  fig.savefig(results_dir / f"aspect_ratio_{aspect_ratio:g}_{name}.pdf")
plt.show()

In [ ]:
fig, _ = plot_convergence(history=history, max_gel_fraction=max_gel_fraction)
fig.savefig(results_dir / f"aspect_ratio_{aspect_ratio:g}_convergence.pdf")
plt.show()

In [ ]:
figures = plot_design_evolution(
  setup=setup, history=history, fractions=(0.0, 0.2, 0.4, 0.6, 1.0)
)
for epoch, (fig, _) in figures.items():
  fig.savefig(
    results_dir / f"aspect_ratio_{aspect_ratio:g}_evolution_epoch_{epoch}.pdf",
    pad_inches=0.0,
  )
plt.show()

### Ablation - aspect ratio

In [ ]:
aspect_ratios = (200.0,)  # (60.0, 80.0, 100.0, 150.0, 200.0) #
max_gel_fraction = 0.4
for aspect_ratio in aspect_ratios:
  settings = {**baseline_settings, "aspect_ratio": aspect_ratio}
  history = run_optimization(
    setup=build_setup(**settings),
    max_gel_fraction=max_gel_fraction,
    monitor=True,
  )
  np.savez(results_dir / f"aspect_ratio_{aspect_ratio:g}.npz", **history, **settings)

In [ ]:
final_errors = []
for aspect_ratio in aspect_ratios:
  history = dict(np.load(results_dir / f"aspect_ratio_{aspect_ratio:g}.npz"))
  setup = build_setup(**{**baseline_settings, "aspect_ratio": aspect_ratio})
  figures = plot_final_state(
    setup=setup, design=history["design"][-1], disp=history["disp"][-1]
  )
  figures["shape"][1].set_title(rf"$L/h = {aspect_ratio:g}$")
  for name, (fig, _) in figures.items():
    fig.savefig(results_dir / f"aspect_ratio_{aspect_ratio:g}_{name}.pdf")
  final_errors.append(np.exp(history["objective"][-1]))
  plt.show()


### Ablation - num elements

In [ ]:
ablation_key, ablation_values = "num_elems", (100, 150, 200, 300, 400)
for value in ablation_values:
  settings = {**baseline_settings, ablation_key: value}
  history = run_optimization(
    setup=build_setup(**settings), max_gel_fraction=max_gel_fraction
  )
  np.savez(results_dir / f"{ablation_key}_{value:g}.npz", **history, **settings)

### Ablation - num fibers

In [ ]:
ablation_key, ablation_values = "num_fibers", (4, 6, 8, 12, 16)

In [ ]:
ablation_label = r"Number of elements $N_e$"
final_errors, epoch_times = [], []
for value in ablation_values:
  history = dict(np.load(results_dir / f"{ablation_key}_{value:g}.npz"))
  setup = build_setup(**{**baseline_settings, ablation_key: value})
  figures = plot_final_state(
    setup=setup, design=history["design"][-1], disp=history["disp"][-1]
  )
  figures["shape"][1].set_title(f"{ablation_label} = {value:g}")
  for name, (fig, _) in figures.items():
    fig.savefig(results_dir / f"{ablation_key}_{value:g}_{name}.pdf")
  plt.show()
  final_errors.append(np.exp(history["objective"][-1]))
  epoch_times.append(np.median(history["time_epoch"][1:]))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(7.0, 3.0))
axes[0].semilogy(ablation_values, final_errors, color="k", marker="o")
axes[0].set_ylabel("Relative curvature error")
axes[1].loglog(ablation_values, epoch_times, color="k", marker="o")
axes[1].set_ylabel("Time per epoch (s)")
for ax in axes:
  ax.set_xlabel(ablation_label)
fig.tight_layout()
fig.savefig(results_dir / f"{ablation_key}_summary.pdf")
plt.show()